In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "buttelmann2017great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "FBHelpingApes_Complete_DataForJosep_rev_table1.csv")
complete_path_2 = os.path.join(original_data_pathway, "FBHelpingApes_Complete_DataForJosep_rev_table2.csv")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

# df1 = pd.read_csv(complete_path_1)

df = pd.read_csv(complete_path_2)

df['study_id']="buttelmann2017great"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [3]:

df.rename(columns={"study": "experiment",
                    "subject":"participant",
                    "sex":"sex_original",
                    "species":"species_original",
                    "age in years":"age",
                    "condition in study":"condition",
                    "trial number":"trial"}, inplace=True)

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

df.columns = df.columns.str.replace(' ', '_', regex=True)
df['experiment'] = df['experiment'].astype(str)


In [4]:
response_list= [[0, 'empty_box'],
                [1,'box_with_object'],
                [999,'']]
for x,y in response_list:
    df.loc[df.response == x, ['response_codes']] = y

df['response'].replace(999, np.nan, inplace=True)

condition_list = [['i','ignorance'],
                ['fb2', 'false_belief_exp2'],
                ['tb','true_belief'],
                ['fb','false_belief']
                ]
for x,y in condition_list:
    df['condition'].replace(x, y, inplace=True)

# df.columns
    
df.rename(columns={"age":"age_in_years"}, inplace=True)

In [5]:
fulldf=df[['study_id','experiment', 'participant', 'age_in_years','sex', 'species',  
       'trial','condition',  'response_codes']]

fulldf.columns =fulldf.columns.str.replace('_codes', '')

for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'buttelmann2017great_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'buttelmann2017great_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
